# Web-search RAG with Haystack and Scavio

This notebook builds a **web-search RAG pipeline**: it answers a question using
*live* web results instead of a static document store. The retrieval step is
[Scavio](https://scavio.dev), a unified Search API for AI agents, exposed to
[Haystack](https://haystack.deepset.ai) through the `ScavioWebSearch` component.
It is a free alternative to paid web-search / answer-engine tools such as
Tavily, Exa, or SerpAPI.

The pipeline wires three components:

`ScavioWebSearch` (live Google results as Documents) -> `PromptBuilder`
(grounds a prompt in those Documents) -> `OpenAIGenerator` (`gpt-4o-mini`
writes the answer).

## Prerequisites

```bash
pip install scavio-haystack haystack-ai openai python-dotenv
```

Environment variables (a `.env` file in the cookbook root works too):

- `SCAVIO_API_KEY` - free key at [dashboard.scavio.dev](https://dashboard.scavio.dev)
- `OPENAI_API_KEY` - for `gpt-4o-mini`


In [1]:
# Install dependencies (uncomment to run inside the notebook)
# %pip install scavio-haystack haystack-ai openai python-dotenv

In [2]:
from dotenv import load_dotenv
from haystack import Pipeline
from haystack.components.builders import PromptBuilder
from haystack.components.generators import OpenAIGenerator
from haystack.utils import Secret

from haystack_integrations.components.websearch.scavio import ScavioWebSearch

# Load SCAVIO_API_KEY and OPENAI_API_KEY from the cookbook .env
load_dotenv(override=True)

False

## Build the web-search component

`ScavioWebSearch` reads `SCAVIO_API_KEY` from the environment. Its `run(query=...)`
returns `documents` (a list of Haystack `Document` objects, each with `title` and
`url` metadata) and `links` (the list of source URLs).

In [3]:
web_search = ScavioWebSearch(
    api_key=Secret.from_env_var("SCAVIO_API_KEY"),
    top_k=5,
)

# Quick look at what retrieval returns on its own
preview = web_search.run(query="What is Haystack by deepset?")
print(f"Retrieved {len(preview['documents'])} documents")
for doc in preview["documents"]:
    print("-", doc.meta["title"], "->", doc.meta["url"])

Retrieved 5 documents
- Haystack | Haystack -> https://haystack.deepset.ai/
- deepset-ai/haystack: Open-source AI orchestration ... -> https://github.com/deepset-ai/haystack
- Introducing Haystack Enterprise Starter | deepset Blog -> https://www.deepset.ai/blog/introducing-haystack-enterprise
- deepset AI -> https://www.deepset.ai/
- Haystack Enterprise Platform Documentation -> https://docs.cloud.deepset.ai/docs/deepset-platform-and-haystack


## Build the RAG pipeline

The `PromptBuilder` template iterates over the retrieved `documents` to ground the
answer, and the `OpenAIGenerator` produces the final reply. We connect
`search.documents` into the prompt builder, and the prompt builder into the LLM.

In [4]:
template = """\
Given the web search results below, answer the question concisely and factually.
Use only the information in the results. If the answer is not present, say so.

Results:
{% for doc in documents %}
[{{ loop.index }}] {{ doc.meta["title"] }} ({{ doc.meta["url"] }})
{{ doc.content }}
{% endfor %}

Question: {{ query }}
Answer:"""

pipe = Pipeline()
pipe.add_component("search", web_search)
pipe.add_component("prompt_builder", PromptBuilder(template=template, required_variables=["documents", "query"]))
pipe.add_component("llm", OpenAIGenerator(model="gpt-4o-mini"))

pipe.connect("search.documents", "prompt_builder.documents")
pipe.connect("prompt_builder", "llm")
print("Pipeline ready")

Pipeline ready


## Run it on a live question

The same `query` is passed to both the search component and the prompt builder.
The answer is grounded in whatever the web returns at run time.

In [5]:
query = "What is Haystack by deepset and what is it used for?"

result = pipe.run(data={"search": {"query": query}, "prompt_builder": {"query": query}})

print("Question:", query)
print()
print("Answer:")
print(result["llm"]["replies"][0])

Question: What is Haystack by deepset and what is it used for?

Answer:
Haystack by deepset is an open-source AI orchestration framework designed for building modular and customizable production-ready AI applications in Python. It enables developers to create agentic, context-engineered AI systems and design pipelines and workflows for large language models (LLM) applications.
